# P1-L5｜执行可行性与成本

> **对应课程**：第 22–25 课  
> **模块**：P1-M5 · 市场微观结构与执行  
> **数据身份**：固定随机种子模拟教学数据，不是真实证券行情。

从订单簿和撮合规则推导成交价格、滑点、费用和容量后的净收益。

本实验向最终策略报告提供：**成交规则、成本、容量与可执行性**。

In [1]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "fixtures").exists():
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data" / "fixtures"
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RANDOM_SEED = 365
print("项目:", ROOT)
print("数据:", DATA_DIR)
print("数据身份: 固定随机种子生成的模拟教学数据，不是真实证券行情")


项目: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p1-quant-foundations
数据: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p1-quant-foundations/data/fixtures
数据身份: 固定随机种子生成的模拟教学数据，不是真实证券行情


## 从订单簿得到规模相关成本

对多个下单量穿透卖盘，计算 VWAP 和相对中间价滑点，再判断毛信号是否覆盖成本。

In [2]:
book = pd.read_csv(DATA_DIR / "order_book.csv")
bids = book.loc[book["side"] == "bid"].sort_values("price", ascending=False)
asks = book.loc[book["side"] == "ask"].sort_values("price")
mid = (bids.iloc[0]["price"] + asks.iloc[0]["price"]) / 2.0

def execute_buy(quantity):
    remaining, notional = quantity, 0.0
    for row in asks.itertuples():
        take = min(remaining, int(row.quantity))
        notional += take * row.price
        remaining -= take
        if remaining == 0:
            return notional / quantity
    return np.nan

gross_alpha_bps = 18.0
fee_bps = 2.0
rows = []
for quantity in [100, 500, 1_000, 1_800]:
    vwap = execute_buy(quantity)
    slippage_bps = (vwap / mid - 1.0) * 10_000 if np.isfinite(vwap) else np.nan
    rows.append({
        "quantity": quantity,
        "vwap": vwap,
        "slippage_bps": slippage_bps,
        "net_alpha_bps": gross_alpha_bps - fee_bps - slippage_bps,
    })
display(pd.DataFrame(rows))

,quantity,vwap,slippage_bps,net_alpha_bps
0,100,100.010000,1.000000,15.000000
1,500,100.014600,1.460000,14.540000
2,1000,100.020600,2.060000,13.940000
3,1800,100.028111,2.811111,13.188889


## 验收

- 关键计算能由小例子或独立公式核对；
- 数据时点、信号时点和成交时点没有混用；
- 结果同时包含金融含义和失败边界；
- 产物能够进入第 29 课策略报告和第 30 课能力矩阵。